# Ranking V2 retriever training (Colab)
Runtime -> Change runtime type -> **T4 GPU** before running.

This runs the leakage-safe Track B protocol from the `ranking-v2` branch: **inner-validation**
(finds a fixed epoch budget E*, checkpoint discarded) -> **R0** (fresh init, trains on Hcore,
generates candidates for y_train) -> **R1** (fresh init, trains on Hcore+y_train, generates
candidates for y_val). Run the cells in order -- step 2 needs a number you read off step 1's
output by hand.

`scripts/train_retriever_v2.py` never touches the shared `experiments/results/summary.csv`
(results go to `experiments/results/ranking_v2_retriever_results.csv` instead) and never imports
`scripts/train.py` -- this cannot interfere with any other training queue.

In [ ]:
# Cell 1: mount Drive, clone the ranking-v2 BRANCH (not main), install, link experiments/ + data into Drive, self-check
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/xi2618zh-s/spatial-graph-recommendation.git"
BRANCH = "ranking-v2"
DRIVE_EXPERIMENTS = "/content/drive/MyDrive/sgr_experiments"
DRIVE_DATA = "/content/drive/MyDrive/sgr_data"

!git clone --branch {BRANCH} {REPO}
%cd spatial-graph-recommendation
!git branch --show-current
!pip -q install torch-geometric faiss-cpu

!mkdir -p {DRIVE_EXPERIMENTS}/logs {DRIVE_EXPERIMENTS}/results {DRIVE_DATA}
!rm -rf experiments/logs experiments/results
!ln -s {DRIVE_EXPERIMENTS}/logs experiments/logs
!ln -s {DRIVE_EXPERIMENTS}/results experiments/results

# reuse cached processed data from Drive if you've uploaded it before; otherwise Cell 1b builds it
!mkdir -p data/processed
!cp -n {DRIVE_DATA}/*.csv {DRIVE_DATA}/*.pkl data/processed/ 2>/dev/null || true

import os
for p in ('experiments/logs', 'experiments/results'):
    assert os.path.islink(p) and os.path.realpath(p).startswith('/content/drive'), f"{p} is not Drive-backed"
print("self-check OK: experiments/logs and experiments/results are Drive-backed")
print("data/processed contents:", os.listdir('data/processed'))

In [ ]:
# Cell 1b (only if data/processed/ came up empty above): build processed artifacts and cache to Drive.
# Needs data/raw/loc-gowalla_totalCheckins.txt.gz uploaded to Drive at DRIVE_DATA/raw/ beforehand,
# OR upload it directly into this Colab session at data/raw/ (slower every session, but works).
!mkdir -p data/raw
!cp -n {DRIVE_DATA}/raw/*.gz data/raw/ 2>/dev/null || true
!python scripts/prepare_data.py
!cp -n data/processed/*.csv data/processed/*.pkl {DRIVE_DATA}/

In [ ]:
# Cell 2: STEP 1 -- inner-validation run (train on H_inner, early-stop against v0).
# Safe to interrupt and re-run with --resume; picks up from last.ckpt.
!python scripts/train_retriever_v2.py \
    --config configs/ranking_v2/retriever_inner_validation.yaml \
    --snapshot inner_validation --resume

In [ ]:
# Cell 3: read E* -- STOP and look at this cell's output before running Cell 4.
!cat experiments/logs/v2_retriever_inner_validation/DONE

In [ ]:
# Cell 4: STEP 2 -- R0. Set E_STAR to the number from Cell 3's output (e.g. "best_epoch=240" -> 240),
# then run this cell. Fresh init, fixed E* epochs, no early-stopping decision made in this run.
E_STAR = None  # <-- EDIT THIS before running, e.g. E_STAR = 240
assert E_STAR is not None, "set E_STAR from Cell 3's DONE file output first"
!python scripts/train_retriever_v2.py \
    --config configs/ranking_v2/retriever_r0.yaml \
    --snapshot r0 --epochs {E_STAR} --resume

In [ ]:
# Cell 5: STEP 3 -- R1. Same E_STAR as Cell 4 (do not re-read a new one). Fresh init on Hcore+y_train.
!python scripts/train_retriever_v2.py \
    --config configs/ranking_v2/retriever_r1.yaml \
    --snapshot r1 --epochs {E_STAR} --resume

In [ ]:
# Cell 6: sanity check -- both R0 and R1 should show a DONE file and a best.pt.
!ls -la experiments/logs/v2_retriever_r0/
!ls -la experiments/logs/v2_retriever_r1/
!cat experiments/results/ranking_v2_retriever_results.csv